# Carbon Clock
A machine learning project that uses data from the PJM grid, which serves New Jersey, to predict which hours use the cleanest sources of energy so people know when to run dishwashers, laundry, chargers, and other energy-intensive tasks. Meant for environmental club members at my school and local families.

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error 

## 1. Loading, Preparing, and Checking the Data
Loading hourly CO2 data for the PJM grid from the U.S. Energy Information Administration (EIA) Hourly Electric Grid Monitor, converting from xlsx to csv, reformatting dataframe, converting from UTC time to Eastern Time,  finding where CO2 data begins, and checking for strange min/max/mean values.

**Why we convert xlsx to csv:** the original Excel file is too big to put on GitHub (it is ~110 MB, GitHub only accepts files <= 100 MB). Moreover, the Excel file has over 100 columns, but we only need 2: the time and the CO2 intensity.

**Where to find original xlsx file:** EIA Hourly Electric Grid Monitor --> Download Data --> Balancing Authority/Regional Files --> PJM

In [10]:
raw_data = pd.read_excel("PJM.xlsx", sheet_name = "Published Hourly Data") # create a dataframe from the excel file
raw_data[["UTC time", "CO2 Emissions Intensity for Consumed Electricity"]].to_csv("pjm_carbon.csv", index=False) # save the dataframe to a csv file
print("Saved pjm_carbon.csv")

Saved pjm_carbon.csv


In [ ]:
df = pd.read_csv("pjm_carbon.csv") # create a dataframe from the csv file
df.columns = ["time", "carbon intensity"] # rename the columns so they are easier to work with

df["time"] = pd.to_datetime(df["time"]) # convert text into datetime object
df = df.set_index("time").sort_index() # set time as the index and sort chronologically
df.index = df.index.tz_localize("UTC").tz_convert("America/New_York") # use Eastern Time Zone because it contains the region we are working with (New Jersey)

A lot of the columns relating to CO2 emissions are NaN, so now finding where the EIA begins collecting data.

In [15]:
print("First hour with CO2 data: ", df["carbon intensity"].first_valid_index())
print("Last hour with CO2 data: ", df["carbon intensity"].last_valid_index())
df[df["carbon intensity"].notna()].head()

First hour with CO2 data:  2018-07-01 04:00:00-04:00
Last hour with CO2 data:  2026-11-12 00:00:00-05:00


,carbon intensity
time,
2018-07-01 04:00:00-04:00,0.942599
2018-07-01 05:00:00-04:00,0.925684
2018-07-01 06:00:00-04:00,0.923685
2018-07-01 07:00:00-04:00,0.921855
2018-07-01 08:00:00-04:00,0.952627


In [19]:
df.describe() # look at the data statistics

,carbon intensity
count,71275.000000
mean,0.757337
std,0.099547
min,0.459549
25%,0.681242
50%,0.749586
75%,0.827903
max,1.097955


In [20]:
print("Duplicate hours: ", df.index.duplicated().sum()) # check for duplicate values

Duplicate hours:  0


**Data checks out** because there are no strange/implausible values for each statistic, and there are no duplicate hours.